# COVID-19 Global Impact Dashboard · EDA walkthrough

This notebook runs on the official OWID source when reachable or a CSV supplied by the user. It does not use generated epidemiological observations.

In [ ]:
from pathlib import Path
import pandas as pd
from data_processing import fetch_owid, clean_data
from analysis import engineer_metrics, country_summary, generate_insights


## Load source data
If offline, set `csv_path` to an OWID-compatible local file.

In [ ]:
csv_path = None  # e.g. "../data/owid-covid-data.csv"
raw = pd.read_csv(csv_path) if csv_path else fetch_owid()
print(raw.shape)
raw.head()

## Data quality profile
Invalid dates are removed; duplicates keep the last row per location-date; invalid counts become missing. Missing values remain missing.

In [ ]:
print(raw.dtypes)
missing = raw.isna().mean().sort_values(ascending=False)
missing.head(20)

In [ ]:
data = engineer_metrics(clean_data(raw))
print(data.shape, data.date.min(), data.date.max(), data.location.nunique())
data.head()

## Summary statistics and country comparisons

In [ ]:
numeric = [c for c in ["new_cases", "new_deaths", "total_cases", "total_deaths", "people_vaccinated_per_hundred"] if c in data]
data[numeric].describe().T

In [ ]:
summary = country_summary(data)
summary.head(10)

## Automated findings
Findings are calculated from the chosen rows. Associations are descriptive, not causal.

In [ ]:
for finding in generate_insights(data):
    print("•", finding)

## Optional Plotly exploration

In [ ]:
import plotly.express as px
if "new_cases_7d_avg" in data:
    px.line(data[data.location.isin(data.location.drop_duplicates().head(5))], x="date", y="new_cases_7d_avg", color="location", title="7-day average reported new cases").show()